# Modelado — población censal y fracción sin seguro

Este notebook orquesta `src.modeling.ablacion_poblacion_sin_seguro`. Compara cuatro variantes predefinidas (base, población fija de 2017, fracción sin seguro fija de 2017 y ambas) para regresión de casos y clasificación directa, en h=2 y h=4. Mantiene los mismos cortes y filas de los experimentos anteriores. Las diferencias de 2024 son exploratorias porque esa temporada ya se había inspeccionado.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.modeling.ablacion_poblacion_sin_seguro import ejecutar_ablacion_poblacion_sin_seguro

reporte = ejecutar_ablacion_poblacion_sin_seguro()

h=2: 30160 filas comunes, cuatro variantes predefinidas
h=4: 30030 filas comunes, cuatro variantes predefinidas
MLflow: run aee55538f46c43d491af9a9188095b9f en 'denguekay/ablacion_poblacion_sin_seguro' (sqlite:///C:/Users/Rosa/RufoEsEternoElRegreso/Denguekay/mlflow.db)


In [2]:
import pandas as pd

filas = []
for h, bloque in reporte["horizontes"].items():
    for variante, resultado in bloque["variantes"].items():
        validacion = resultado["folds_validacion"]
        prueba = next(f for f in resultado["folds_prueba"] if f["tipo"] == "prueba_principal")
        filas.append({
            "h": int(h), "variante": variante,
            "F1_validacion_clasificacion": sum(f["clasificacion"]["alerta_directa"]["f1"] for f in validacion) / len(validacion),
            "F1_2024_clasificacion": prueba["clasificacion"]["alerta_directa"]["f1"],
            "VP_2024": prueba["clasificacion"]["alerta_directa"]["vp"],
            "FP_2024": prueba["clasificacion"]["alerta_directa"]["fp"],
        })
pd.DataFrame(filas).sort_values(["h", "variante"]).round(3)

,h,variante,F1_validacion_clasificacion,F1_2024_clasificacion,VP_2024,FP_2024
0,2,base_6,0.598,0.636,811,881
1,2,poblacion_2017,0.659,0.643,808,848
3,2,poblacion_y_sin_seguro_2017,0.655,0.646,808,838
2,2,sin_seguro_2017,0.639,0.662,763,684
4,4,base_6,0.557,0.614,768,877
5,4,poblacion_2017,0.581,0.623,786,879
7,4,poblacion_y_sin_seguro_2017,0.588,0.620,782,882
6,4,sin_seguro_2017,0.590,0.621,757,823


La interpretación, las comparaciones pareadas y los límites se documentan en `docs/modeling/ablacion_poblacion_sin_seguro.md`. El JSON conserva métricas de regresión, alertas y resultados de 2025.